# PII Masking: DeBERTa Fine-Tuning and LLaMA Zero-Shot

1. Inspect the provided NER data.
2. Fine-tune DeBERTa for token classification over `O`, `B-PER`, `I-PER`, and `B-EMAIL`.
3. Evaluate on the test set with accuracy, precision, recall, F1, FPR, and FNR.
4. Run LLaMA 1B as a zero-shot baseline through prompting and JSON parsing.
5. Compare both approaches and analyze results.

Model IDs:

- DeBERTa: `microsoft/deberta-v3-base`
- LLaMA 1B: `meta-llama/Llama-3.2-1B-Instruct`


### DeBERTa 

In [1]:
from pathlib import Path
import json
import numpy as np
from collections import Counter

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
MODEL_PATH     = r"D:/Coding/Models/DeBERTa-v3-base"
OUTPUT_DIR     = r"D:/Coding/Models/DeBERTa-v3-finetuned"   
LOG_DIR        = r"D:/Coding/Models/logs"

# ── 1. Data Preparation ────────────────────────────────────────────────────────
# Use the modified dataset you created in prepare_data.ipynb 
def load_data():
    with open('../datasets/train_data_modified.json', 'r', encoding='utf-8') as file:
        train_data = json.load(file)
    
    # IMPORTANT: Ensure you have also augmented your test data! [cite: 53, 59]
    with open('../datasets/test_data.json', 'r', encoding='utf-8') as file:
        test_data = json.load(file)

    return train_data, test_data

train_data, test_data = load_data()

# Define full label list including I-EMAIL [cite: 30]
label_list = ["O", "B-PER", "I-PER", "B-EMAIL", "I-EMAIL"]
label2id   = {label: i for i, label in enumerate(label_list)}
id2label   = {i: label for label, i in label2id.items()}

full_dataset = Dataset.from_list(train_data)
split_dataset = full_dataset.train_test_split(test_size=0.15, seed=42)
train_ds = split_dataset["train"]
validation_ds = split_dataset["test"]
test_ds = Dataset.from_list(test_data)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# ── 2. Tokenization & Alignment ───────────────────────────────────────────────
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True, max_length=512, is_split_into_words=True)

    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned = []
        prev_word_id = None

        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)
            elif word_id != prev_word_id:
                # Map string tags to IDs [cite: 8, 19]
                label = labels[word_id]
                aligned.append(label2id.get(label, 0)) 
            else:
                aligned.append(-100) 
            prev_word_id = word_id

        all_labels.append(aligned)

    tokenized["labels"] = all_labels
    return tokenized

train_tok = train_ds.map(tokenize_and_align_labels, batched=True, remove_columns=train_ds.column_names)
val_tok = validation_ds.map(tokenize_and_align_labels, batched=True, remove_columns=validation_ds.column_names)
test_tok = test_ds.map(tokenize_and_align_labels, batched=True, remove_columns=test_ds.column_names)

# ── 3. Model & Metrics ────────────────────────────────────────────────────────
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_PATH,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# Use seqeval for entity-level metrics as required by task 
metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# ── 4. Training Arguments ─────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3, # Reduced to prevent immediate overfitting
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True, # Use fp16 instead of bf16 for broader compatibility and stability
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Training data loaded with 28516 lines.
Test data loaded with 3650 lines.


[transformers] The tokenizer you are loading from 'D:/Coding/Models/DeBERTa-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Map:   0%|          | 0/24238 [00:00<?, ? examples/s]

Map:   0%|          | 0/4278 [00:00<?, ? examples/s]

Map:   0%|          | 0/3650 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForTokenClassification LOAD REPORT from: D:/Coding/Models/DeBERTa-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from differe

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.000000,nan,0.782877,0.884804,0.830726,0.884804


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 